In [ ]:
# =============================================================================
# Zelle 01 – Setup Phase 5 (Notebook 06: Agent und Modellvergleich)
# =============================================================================
# Zweck: Gemeinsamer Kontext für Phase 5 (Tool-Schicht, Bestätigungs-Gate,
#   Testset, Varianten A/B/C, Evaluation).
# Begründung: Setup über playlist_agent.projekt (Entscheidung 21.11.2026),
#   keine kopierte Setup-Zelle.
# Datenschutz (P11): echte Werte nur in data/ und reports/errors/.
# =============================================================================

from functools import partial

import numpy as np
import pandas as pd

from playlist_agent import projekt
from playlist_agent.viz_config import stil_aktivieren

K = projekt.kontext()
PROJECT_ROOT, cfg, SEED = K.wurzel, K.cfg, K.seed
RAW, INTERIM, GOLD = K.raw, K.interim, K.gold
FIG, TAB, ERR = K.fig, K.tab, K.err
stil_aktivieren()
tabelle_speichern = partial(projekt.tabelle_speichern, ordner=TAB)
privat_speichern = partial(projekt.tabelle_speichern, ordner=ERR)   # Einzelfälle, privat (P11)

for _n, _p in {"RAW": RAW, "INTERIM": INTERIM, "GOLD": GOLD,
               "FIG": FIG, "TAB": TAB, "ERR": ERR}.items():
    print(f"{_n:8s} {'vorhanden' if _p.is_dir() else 'FEHLT'}")

In [ ]:
# =============================================================================
# Zelle 02 – Paket 5.0: tinytag-Version und lesbare Tag-Felder je Titel (Stichprobe)
# =============================================================================
# Zweck: Vor dem Schema der Bibliothek (Rohschicht) feststellen, welche Felder
#   tinytag liefert (Comment, Genre, Bewertung/RATING, Zusatzfelder) – je Titel.
# Begründung: Probelauf vor der vollständigen Roherfassung (Lehre: NB02 Z.12
#   hat beim Lesen nur 6 Felder ausgewählt; Comment/Bewertung fehlen).
# Stichprobe: 5 mp3 + 1 wav + 1 aif zufällig (SEED) und 3 mp3, deren Serato-
#   Comment "sterne" enthält (dort sind Bewertungen zu erwarten).
# Legende: 1 Zeile = 1 Datei, 1 Spalte = 1 Tag-Feld; leer = Feld nicht gefüllt.
#   Zusatzfelder als 'other:<name>' (tinytag 2) bzw. 'extra:<name>' (tinytag 1).
# Rein lesend (P1/P7-Ausnahme Tags). Stick muss eingesteckt sein.
# Datenschutz: Werte nur lokale Anzeige und privat in reports/errors (P11).
# =============================================================================

from importlib.metadata import version

from tinytag import TinyTag

print("tinytag-Version:", version("tinytag"))
STICK = projekt.stick(cfg)

df_tags = pd.read_csv(INTERIM / "02_12_audio_tags.csv", low_memory=False)
df_db = pd.read_csv(INTERIM / "02_08_database_v2.csv", low_memory=False)

_zufall = pd.concat([df_tags[df_tags["endung"] == ".mp3"].sample(5, random_state=SEED),
                     df_tags[df_tags["endung"] == ".wav"].sample(1, random_state=SEED),
                     df_tags[df_tags["endung"] == ".aif"].sample(1, random_state=SEED)])["pfad_relativ"]
_mit_sternen = (df_db[df_db["tcom"].fillna("").str.contains("sterne", case=False)
                      & df_db["schluessel"].str.lower().str.endswith(".mp3")]
                ["schluessel"].sample(3, random_state=SEED))
_probe = list(_zufall) + [str(s).replace("\\", "/") for s in _mit_sternen]


def tag_felder(tag) -> dict:
    """Alle öffentlichen Attribute mit Wert; Zusatzfelder aus 'other' (v2) bzw. 'extra' (v1)."""
    # 'images' ausgeschlossen: Objekt-Text mit Speicheradresse ändert sich je Lauf (keine Information)
    d = {k: v for k, v in vars(tag).items()
         if not k.startswith("_") and k != "images" and v not in (None, "", {}, [])}
    for zusatz in ("other", "extra"):
        for k, v in (d.pop(zusatz, None) or {}).items():
            if v not in (None, "", [], {}):
                d[f"{zusatz}:{k}"] = v
    return d


_zeilen02 = []
for p in _probe:
    try:
        _zeilen02.append({"datei": p.rsplit("/", 1)[-1], **tag_felder(TinyTag.get(STICK / p, image=False))})
    except Exception as e:
        _zeilen02.append({"datei": p.rsplit("/", 1)[-1], "lesefehler": repr(e)})

df_probe = pd.DataFrame(_zeilen02).set_index("datei")
print(f"Stichprobe: {len(df_probe)} Dateien, {df_probe.shape[1]} verschiedene Felder")
print("Felder (gefüllt in n Dateien):", df_probe.notna().sum().sort_values(ascending=False).to_dict())

with pd.option_context("display.max_colwidth", 60, "display.max_columns", None):
    display(df_probe)
_ = privat_speichern(df_probe.reset_index(), "06_02_tag_felder_stichprobe")       # privat

In [ ]:
# =============================================================================
# Zelle 03 – Paket 5.0: Bibliothek Rohschicht – alle Musikstücke, alle Informationen
# =============================================================================
# Zweck: Eine Zeile je Audiodatei (mp3, wav, aif), alle verfügbaren Informationen
#   als Spalten, nichts interpretiert (Rohschicht, Entscheidung 22.11.2026):
#   datei_* / dateityp   Index (Pfad, Ordner, Name, Grösse, Änderungsdatum)
#   sha256, audio_sha256 Prüfsummen (Manifest NB04 Z.08, Audio-Hash NB05 Z.38)
#   audio_*              tinytag: Dauer, Bitrate, Abtastrate, Kanäle, Bittiefe, verlustfrei
#   tag_*                tinytag: alle Tag-Felder inkl. Zusatzfelder ('other')
#   tag_popm_*           Bewertung (ID3-Rahmen POPM), eigener Leser (tinytag liest ihn nicht)
#   tag_rahmen_liste     alle vorhandenen ID3-Rahmen bzw. Chunks (Vollständigkeit)
#   serato_*             Serato database V2 (NB02 Z.08), Zuordnung über NFC-Pfad
# Begründung: vollständige Roherfassung statt Auswahl beim Lesen (Lehre NB02 Z.12).
# Rein lesend, nur Tag-Bereiche, Audiodaten werden übersprungen (P1/P7-Ausnahme).
# Stick muss eingesteckt sein. Ausgabe privat: data/interim/06_03_bibliothek_roh.csv
# =============================================================================

import posixpath
import struct
import time

from tinytag import TinyTag

from playlist_agent.canon import nfc

STICK = projekt.stick(cfg)
AUDIO = cfg["index"]["audio_endungen"]


# --- eigener Leser für ID3-Rahmen / Chunks (POPM und Vollständigkeit) ------------------
def _ss(b: bytes) -> int:                       # synchsafe-Ganzzahl (7 Bit je Byte)
    return (b[0] & 0x7F) << 21 | (b[1] & 0x7F) << 14 | (b[2] & 0x7F) << 7 | (b[3] & 0x7F)


def id3_aus_bytes(b: bytes) -> tuple:
    """ID3v2-Tag (beginnt mit 'ID3') -> (Version, Rahmen-Kennungen, POPM-Einträge)."""
    if len(b) < 10 or b[:3] != b"ID3":
        return None, [], []
    ver, flags, groesse = b[3], b[5], _ss(b[6:10])
    tag = b[10:10 + groesse]
    pos, rahmen, popm = 0, [], []
    if flags & 0x40 and ver >= 3:                                  # erweiterter Kopf
        pos = _ss(tag[0:4]) if ver == 4 else 4 + int.from_bytes(tag[0:4], "big")
    kl = 6 if ver == 2 else 10
    while pos + kl <= len(tag):
        if ver == 2:
            rid, n = tag[pos:pos + 3], int.from_bytes(tag[pos + 3:pos + 6], "big")
        else:
            rid = tag[pos:pos + 4]
            n = _ss(tag[pos + 4:pos + 8]) if ver == 4 else int.from_bytes(tag[pos + 4:pos + 8], "big")
        if not rid.strip(b"\x00") or not rid.isalnum() or n <= 0:
            break                                                  # Auffüllbereich
        rid = rid.decode("latin-1")
        daten = tag[pos + kl:pos + kl + n]
        if rid in ("POPM", "POP"):
            email, _, rest = daten.partition(b"\x00")
            popm.append((email.decode("latin-1"), rest[0] if rest else None))
        rahmen.append(rid)
        pos += kl + n
    return f"2.{ver}", rahmen, popm


def roh_tags(pfad) -> dict:
    """Aufbau, ID3-Version, Liste der Rahmen/Chunks, POPM (mp3: ID3 am Anfang; wav/aif: Chunks)."""
    with open(pfad, "rb") as f:
        kopf = f.read(12)
        if kopf[:3] == b"ID3":
            f.seek(0)
            k = f.read(10)
            ver, rahmen, popm = id3_aus_bytes(k + f.read(_ss(k[6:10])))
            return {"aufbau": "id3_anfang", "ver": ver, "rahmen": rahmen, "popm": popm}
        if kopf[:4] == b"RIFF" and kopf[8:12] == b"WAVE":
            aufbau, endian = "wav_riff", "<"
        elif kopf[:4] == b"FORM" and kopf[8:12] in (b"AIFF", b"AIFC"):
            aufbau, endian = "aif_form", ">"
        else:
            return {"aufbau": "unbekannt", "ver": None, "rahmen": [], "popm": []}
        rahmen, popm, ver = [], [], None
        while True:
            ch = f.read(8)
            if len(ch) < 8:
                break
            cid, n = ch[:4].decode("latin-1"), struct.unpack(endian + "I", ch[4:])[0]
            rahmen.append("CHUNK:" + cid.strip())
            if cid in ("id3 ", "ID3 "):
                ver, r, p = id3_aus_bytes(f.read(n))
                rahmen += r
                popm += p
            elif aufbau == "wav_riff" and cid == "LIST":
                inhalt = f.read(n)
                if inhalt[:4] == b"INFO":
                    i = 4
                    while i + 8 <= len(inhalt):
                        sid, sn = inhalt[i:i + 4].decode("latin-1"), struct.unpack("<I", inhalt[i + 4:i + 8])[0]
                        rahmen.append("INFO:" + sid)
                        i += 8 + sn + (sn % 2)
            else:
                f.seek(n, 1)                                       # Audiodaten überspringen
            if n % 2:
                f.seek(1, 1)                                       # Auffüllbyte
        return {"aufbau": aufbau, "ver": ver, "rahmen": rahmen, "popm": popm}


# --- tinytag: alle Felder -----------------------------------------------------------
AUDIO_FELDER = {"duration", "bitrate", "samplerate", "channels", "bitdepth", "is_lossless", "mime_type"}
WEGLASSEN = {"filename", "filesize", "images"}                     # Pfad/Grösse aus Index


def _wert(v):
    return "; ".join(map(str, v)) if isinstance(v, (list, tuple)) else v


def tinytag_felder(pfad) -> dict:
    t = TinyTag.get(pfad, image=False)
    d = {}
    for k, v in vars(t).items():
        if k.startswith("_") or k in WEGLASSEN or v in (None, "", [], {}):
            continue
        if k == "other":
            for ok, ov in v.items():
                if ov not in (None, "", [], {}):
                    d[f"tag_{ok}"] = _wert(ov)
        else:
            d[("audio_" if k in AUDIO_FELDER else "tag_") + k] = _wert(v)
    return d


# --- Lauf über alle Audiodateien ----------------------------------------------------
df_basis = pd.read_csv(INTERIM / "05_38_audiohash_alle_mp3.csv", low_memory=False)
df_basis = df_basis[df_basis["endung"].isin(AUDIO) & ~df_basis["appledouble"]].copy()
print(f"Audiodateien laut Index: {len(df_basis)}")

_t0, _zeilen, _fehler = time.perf_counter(), [], []
for p in df_basis["pfad_relativ"]:
    z = {"pfad_relativ": p}
    try:
        z.update(tinytag_felder(STICK / p))
    except Exception as e:
        _fehler.append({"pfad_relativ": p, "teil": "tinytag", "fehler": repr(e)})
    try:
        r = roh_tags(STICK / p)
        z.update({"tag_aufbau": r["aufbau"], "tag_id3_version": r["ver"],
                  "tag_rahmen_liste": "; ".join(r["rahmen"]),
                  "tag_popm_kennung": "; ".join(e for e, _ in r["popm"]) or None,
                  "tag_popm_wert": "; ".join(str(w) for _, w in r["popm"]) or None})
    except Exception as e:
        _fehler.append({"pfad_relativ": p, "teil": "rahmen", "fehler": repr(e)})
    _zeilen.append(z)
print(f"Laufzeit: {(time.perf_counter() - _t0) / 60:.1f} min | Lesefehler: {len(_fehler)}")

# --- Zusammenführen: Index + Tags + Serato ------------------------------------------
df_basis = df_basis.rename(columns={"bytes": "datei_bytes", "mtime": "datei_mtime"})
df_basis.insert(1, "dateityp", df_basis["endung"].str.lstrip("."))
df_basis.insert(2, "datei_ordner", df_basis["pfad_relativ"].map(posixpath.dirname))
df_basis.insert(3, "datei_name", df_basis["pfad_relativ"].map(posixpath.basename))
df_basis = df_basis.drop(columns=["endung", "appledouble"])
df_bib = df_basis.merge(pd.DataFrame(_zeilen), on="pfad_relativ", how="left")

SERATO_NAMEN = {"tsng": "titel", "tart": "interpret", "talb": "album", "tgen": "genre",
                "tcom": "comment", "tlen": "dauer", "tsiz": "groesse", "tbpm": "bpm",
                "tadd": "hinzugefuegt", "tkey": "tonart", "ttyr": "jahr", "tlbl": "label",
                "tcmp": "komponist", "tgrp": "gruppe", "trmx": "remix", "ttyp": "dateityp",
                "tbit": "bitrate", "tsmp": "abtastrate", "pfil": "pfad_original"}
df_db = pd.read_csv(INTERIM / "02_08_database_v2.csv", low_memory=False)
df_db["pfad_relativ"] = df_db["schluessel"].map(lambda s: nfc(str(s)).replace("\\", "/"))
_doppelt = df_db["pfad_relativ"].duplicated().sum()
df_db = df_db.drop_duplicates("pfad_relativ").drop(columns=["schluessel"])
df_db.columns = ["pfad_relativ" if c == "pfad_relativ" else f"serato_{SERATO_NAMEN.get(c, c)}"
                 for c in df_db.columns]
df_bib = df_bib.merge(df_db, on="pfad_relativ", how="left")
print(f"Serato: {len(df_db)} Tracks nach NFC-Zusammenführung ({_doppelt} doppelte Schlüssel entfernt) "
      f"| zugeordnet: {df_bib['serato_titel'].notna().sum()} von {len(df_bib)} Dateien")
print(f"Bibliothek: {len(df_bib)} Zeilen, {df_bib.shape[1]} Spalten")

# --- Ausgabe: Ausschnitt, Füllgrad, Speichern -----------------------------------------
with pd.option_context("display.max_columns", None, "display.max_colwidth", 40):
    display(df_bib.head(15))

df_fuell = (df_bib.notna().groupby(df_bib["dateityp"]).mean().T.mul(100).round(1)
            .assign(gesamt=(df_bib.notna().mean() * 100).round(1))
            .rename_axis("spalte").reset_index())
df_fuell.columns = [c if c == "spalte" else f"{c} [%]" for c in df_fuell.columns]
with pd.option_context("display.max_rows", 300):
    display(df_fuell)

_ = projekt.tabelle_speichern(df_bib, "06_03_bibliothek_roh", INTERIM)     # privat
_ = tabelle_speichern(df_fuell, "06_03_bibliothek_fuellgrad")               # nur Spaltennamen + %
if _fehler:
    _ = privat_speichern(pd.DataFrame(_fehler), "06_03_bibliothek_lesefehler")


In [ ]:
# =============================================================================
# Zelle 04 – Paket 5.0: Notizen und Bewertungen in der Bibliothek (Rohschicht)
# =============================================================================
# Zweck: Nutzerangaben 22.11.2026 mit Zahlen belegen:
#   BE (bestes Ende) am Anfang von tag_title und serato_titel;
#   Stimmungswörter in tag_genre und serato_genre; Bewertung in POPM.
#   Gegenprobe Bewertung: Sterne im Comment (tag_comment, serato_comment).
# Legende: BE = Titel beginnt mit optionalem '+', 'BE' und Zeitmarke m:ss/m.ss |
#   popm_wert = Rohwert 0–255 je Programmkennung | sterne_comment = Ziffer vor 'sterne'.
# Datenschutz: Zählwerte in tables/; Einzelfälle privat in reports/errors.
# =============================================================================

import re

BE = re.compile(r"^\s*\+?\s*BE\s*(\d{1,2}[:.]\d{2})", re.IGNORECASE)
STERNE = re.compile(r"([1-5])\s*sterne", re.IGNORECASE)


def _erster(s: pd.Series, rx) -> pd.Series:
    return s.fillna("").astype(str).str.extract(rx)[0]


df_n = df_bib[["pfad_relativ", "dateityp", "tag_title", "serato_titel", "tag_genre", "serato_genre",
               "tag_comment", "serato_comment", "tag_popm_kennung", "tag_popm_wert"]].copy()
df_n["be_tag"] = _erster(df_n["tag_title"], BE)
df_n["be_serato"] = _erster(df_n["serato_titel"], BE)
df_n["sterne_tag_comment"] = _erster(df_n["tag_comment"], STERNE)
df_n["sterne_serato_comment"] = _erster(df_n["serato_comment"], STERNE)

# --- 1 BE -------------------------------------------------------------------------
print("1 BE (bestes Ende) je Dateityp:")
print(pd.DataFrame({"titel_tag [Dateien]": df_n.groupby("dateityp")["be_tag"].count(),
                    "titel_serato [Dateien]": df_n.groupby("dateityp")["be_serato"].count(),
                    "beide [Dateien]": df_n.dropna(subset=["be_tag", "be_serato"]).groupby("dateityp").size()})
      .fillna(0).astype(int).to_string())
_b = df_n.dropna(subset=["be_tag", "be_serato"])
print(f"BE in beiden Quellen: {len(_b)} | gleiche Zeitmarke: "
      f"{(_b['be_tag'].str.replace('.', ':') == _b['be_serato'].str.replace('.', ':')).sum()}")

# --- 2 Genre ------------------------------------------------------------------------
df_genre = (pd.concat([df_n["tag_genre"].value_counts().rename("tag_genre [Dateien]"),
                       df_n["serato_genre"].value_counts().rename("serato_genre [Dateien]")], axis=1)
            .fillna(0).astype(int).rename_axis("genre_wert").reset_index()
            .sort_values("tag_genre [Dateien]", ascending=False, ignore_index=True))
print(f"\n2 Genre: {len(df_genre)} verschiedene Werte "
      f"(tag_genre gefüllt {df_n['tag_genre'].notna().sum()}, serato_genre gefüllt {df_n['serato_genre'].notna().sum()})")
with pd.option_context("display.max_rows", 400):
    display(df_genre)

# --- 3 Bewertung --------------------------------------------------------------------
print(f"\n3 POPM gefüllt: {df_n['tag_popm_wert'].notna().sum()} Dateien")
print("Programmkennung:", df_n["tag_popm_kennung"].value_counts().to_dict())
print("Rohwerte (0–255):", df_n["tag_popm_wert"].value_counts().sort_index().to_dict())
print("\nGegenprobe: POPM-Rohwert × Sterne im Datei-Comment (nur Dateien mit beidem):")
_g = df_n.dropna(subset=["tag_popm_wert", "sterne_tag_comment"])
print(pd.crosstab(_g["tag_popm_wert"], _g["sterne_tag_comment"]).to_string() if len(_g) else "keine")
print("\nSterne im Comment: Datei-Tag", df_n["sterne_tag_comment"].value_counts().sort_index().to_dict(),
      "| Serato", df_n["sterne_serato_comment"].value_counts().sort_index().to_dict())

_ = privat_speichern(df_genre, "06_04_genre_werte")                 # Genre-Werte nur privat (P11)
_ = tabelle_speichern(df_genre.assign(genre_wert="(privat)").head(0), "06_04_genre_platzhalter") \
    if False else None                                  # Genre-Werte nur privat (P11)
_ = privat_speichern(df_genre, "06_04_genre_werte")

In [ ]:
# =============================================================================
# Zelle 05 – Diagnose: gespeicherte vs. aktuelle Einzelfall-Tabelle (Zelle 04)
# =============================================================================
# Zweck: Ursache von ArtefaktGeaendert feststellen, bevor etwas überschrieben wird.
# Vergleich: Zeilen, Spalten (Menge und Reihenfolge), Werte je Spalte.
# Rein lesend.
# =============================================================================

from io import StringIO

_alt = pd.read_csv(ERR / "06_04_notizen_bewertung_je_datei.csv", dtype=str, keep_default_na=False)
_neu = pd.read_csv(StringIO(df_n.to_csv(index=False)), dtype=str, keep_default_na=False)

print(f"Zeilen: gespeichert {len(_alt)} | aktuell {len(_neu)}")
print("Spalten gleich (Menge):", set(_alt.columns) == set(_neu.columns),
      "| Reihenfolge gleich:", list(_alt.columns) == list(_neu.columns))
print("nur gespeichert:", sorted(set(_alt.columns) - set(_neu.columns)))
print("nur aktuell:   ", sorted(set(_neu.columns) - set(_alt.columns)))

if len(_alt) == len(_neu):
    _a = _alt.set_index("pfad_relativ").sort_index()
    _b = _neu.set_index("pfad_relativ").sort_index()
    print("Pfade gleich:", list(_a.index) == list(_b.index))
    if list(_a.index) == list(_b.index):
        _gem = [c for c in _a.columns if c in _b.columns]
        _diff = {c: int((_a[c] != _b[c]).sum()) for c in _gem}
        print("abweichende Werte je Spalte:", {c: n for c, n in _diff.items() if n})
        _c = next((c for c, n in _diff.items() if n), None)
        if _c:
            _bsp = pd.DataFrame({"gespeichert": _a[_c], "aktuell": _b[_c]})[_a[_c] != _b[_c]].head(5)
            print(f"\nBeispiele Spalte '{_c}':")
            display(_bsp)

### Befunde und Entscheidungen 22.11.2026 – Notizen und Bewertung (Bibliothek, Rohschicht)

**BE (bestes Ende, für das Live-Spielen):** nur mp3; im Datei-Titel 423, im Serato-Titel 418; in allen 418 Fällen mit beiden Quellen gleiche Zeitmarke. -> Merkmal `be_zeit`, für Titelvergleiche aus dem Titel entfernen; für die Suche nur Hinweis.

**Stimmung:** steht im Genre (Datei-Tag 4.692 gefüllt, Serato 3.630), als zusammengesetzter Name aus Stimmung + Stil + Zusatz (z. B. Soft|Techno|Beat). Zerlegung in Wörter (auch an Grossbuchstaben). Rollen (Nutzer):
- Stimmung: soft, push, happy, down, rush, goodfly/fly, rock (rock nur zusammen mit einem Stil-Wort, sonst Stil)
- Stil: techno, tech, house, techhouse, elektro, electro, electronic, electronica, minimal, ambient, lounge, dub, acid, funk, trance, r&b u. a.
- Zusatz: beat | Arbeitsnotiz: prüfen, schneiden, intro | keine Information: Other, Andere, Unbekanntes Genre, (255), defekte Zeichen

**Bewertung:** POPM in 2.023 Dateien (Windows Media Player 9: 1.697 + 6 abgeschnittene Kennungen; unknown@unknown: 311). Skala (bestätigt): 1/64/128/196/255 -> 1–5 Sterne; 153/154 -> 3, 194/204/205 -> 4. Gegenprobe mit Sternen im Comment: 1.188 von 1.236 gleich (96,1 %).
- Eine Quelle oder alle gleich -> übernommen, Herkunft dokumentiert.
- Widerspruch zwischen Quellen -> keine automatische Wahl; Nutzer entscheidet neu, Entscheidung als Label gespeichert (analog P13).

In [ ]:
# Zelle 06 – Merkmalsschicht: Masterliste aller Musikstuecke
# Zweck: Rohschicht (Z.03) + abgeleitete Merkmale nach den Regeln vom 22./24.11.2026
#        (bibliothek.merkmale). Rohspalten bleiben unveraendert, jedes Merkmal hat Herkunft.
# Begruendung: Wahrheit = Rohwerte + Nutzerlabels; Merkmale sind Messungen und werden
#        deshalb mit Einzelfaellen ausgegeben (Nutzer prueft).
# Ausgabe: Zaehlungen je Merkmal; Masterliste privat in data/processed;
#        Einzelfaelle privat in reports/errors (Pfad wird beim Speichern gemeldet).
import pandas as pd
from playlist_agent import bibliothek, projekt

K = projekt.kontext()
PROCESSED = K.wurzel / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

# alles als Text lesen: POPM-Werte sonst als Float ('196.0'), Pfade bleiben unveraendert
df_roh = pd.read_csv(K.interim / "06_03_bibliothek_roh.csv", dtype=str, low_memory=False)
kuerzel = bibliothek.lade_kuerzel(K.gold / "06_05_interpret_kuerzel.csv")
df_bib = bibliothek.merkmale(df_roh, kuerzel)

neu = [s for s in df_bib.columns if s not in df_roh.columns]
print(f"Zeilen {len(df_bib)} | Spalten roh {df_roh.shape[1]} + Merkmale {len(neu)} = {df_bib.shape[1]}")
print(f"Kuerzel-Liste: {len(kuerzel)} Eintraege")
print(f"Rohspalten unveraendert: {df_bib[df_roh.columns].equals(df_roh)}")

# --- Zaehlungen ---
def zaehle(spalte):
    print(f"\n{spalte}:")
    print(df_bib[spalte].value_counts(dropna=False).to_string())

for s in ["titel_quelle", "be_markiert", "genre_quelle", "genre_keine_info", "genre_defekt",
          "bewertung", "bewertung_herkunft", "bewertung_konflikt", "popm_abweichung",
          "dj_herkunft", "interpret_herkunft"]:
    zaehle(s)
print(f"\nmit BE-Zeit: {df_bib['be_zeit'].notna().sum()} | mit BA-Zeit: {df_bib['ba_zeit'].notna().sum()}")
print(f"mit Stimmung: {df_bib['stimmung'].notna().sum()} | mit Stil: {df_bib['stil'].notna().sum()}")

# --- Masterliste (privat) ---
_ = projekt.tabelle_speichern(df_bib, "06_06_bibliothek", ordner=PROCESSED)

# --- Einzelfaelle (privat) ---
basis = ["pfad_relativ", "titel_quelle", "tag_title", "serato_titel"]
_ = projekt.tabelle_speichern(
    df_bib.loc[df_bib["be_markiert"] | df_bib["ba_zeit"].notna(),
               basis + ["be_zeiten", "ba_zeiten", "be_markiert"]],
    "06_06_be_ba_faelle", ordner=K.err)
_ = projekt.tabelle_speichern(
    df_bib.groupby(["genre_quelle", df_bib["tag_genre"].fillna(df_bib["serato_genre"]).rename("genre"),
                    "stimmung", "stil", "genre_notiz", "genre_keine_info", "genre_defekt"],
                   dropna=False).size().rename("anzahl [Dateien]").reset_index(),
    "06_06_genre_zerlegung", ordner=K.err)
_ = projekt.tabelle_speichern(
    df_bib.loc[df_bib["bewertung_konflikt"] | df_bib["popm_abweichung"],
               ["pfad_relativ", "tag_comment", "serato_comment", "tag_popm_wert",
                "bewertung_werte", "bewertung", "bewertung_konflikt", "popm_sterne", "popm_abweichung"]],
    "06_06_bewertung_auffaellig", ordner=K.err)
_ = projekt.tabelle_speichern(
    df_bib[["pfad_relativ", "datei_ordner", "serato_interpret", "tag_artist",
            "dj", "dj_herkunft", "interpret", "interpret_herkunft"]],
    "06_06_dj_interpret", ordner=K.err)

print(df_bib["jahr_herkunft"].value_counts().to_string())
print(f"\nmit Jahr: {df_bib['jahr'].notna().sum()} | nur Jahrzehnt: {df_bib['jahrzehnt'].notna().sum()}")
print(df_bib["jahr"].describe().to_string())

### Befunde und Entscheidungen 5.0 – Bibliothek (Merkmalsschicht), 24.11.2026

**Ergebnis:** Masterliste `data/processed/06_06_bibliothek.csv` (privat): 5.236 Audiodateien,
114 Rohspalten (unveraendert) + 22 abgeleitete Merkmale mit Herkunft.

**Regeln (Entscheidungen 22./24.11.2026)**
- BE = bestes Ende, BA = bester Anfang: Zeit (m:ss, mm:ss, h:mm:ss) irgendwo im Titel, mehrere moeglich;
  Marker BE/be auch ohne Zeit. Titel aus Tag, sonst Serato.
- Genre: Stimmung = soft, push, happy, down, rush, goodfly, fly, beat, rock (rock nur mit Stil-Wort);
  Stil = uebrige Woerter; TechHouse = ein Stil; Kauf-CD-Genres "Classic Rock"/"Folk-Rock" = Stil rock;
  Arbeitsnotizen (pruefen, schneiden, intro) eigene Spalte.
- Bewertung: nur Comment (Tag, Serato); Einigkeit -> Wert, Widerspruch -> Konflikt (Nutzer entscheidet, P13).
  POPM (ID3v2 Popularimeter, vom Nutzer nie gepflegt) nur Gegenprobe.
- DJ (spielt das Stueck im Set) getrennt vom Interpret: Serato "<Kuerzel> Mix" -> Titel -> Ordner -> Datei;
  Kuerzel-Liste privat in data/gold. Interpret nur aus dem Tag.
- Plus-Zeichen am Titelanfang: Bedeutung unbekannt -> nicht ausgewertet.

**Kennzahlen [Dateien]**
| Merkmal | Anzahl |
|---|---|
| BE-Zeit / BA-Zeit / BE markiert | 520 / 19 / 585 |
| Stimmung / Stil | 597 / 3.941 |
| Bewertung (3/4/5/2 Sterne) | 1.487 (142 / 796 / 547 / 2) |
| Bewertungskonflikt / POPM-Abweichung | 20 / 45 |
| DJ (serato_mix / ordner / titel / datei) | 1.918 (1.653 / 170 / 94 / 1) |
| Interpret aus Tag | 2.455 |

**Belegstatus**
- Genre-Zerlegung: Vollpruefung durch Nutzer (208 Genre-Namen), Korrektur eingearbeitet.
- BE/BA, DJ-Herkunft: keine

In [ ]:
# Zelle 07 – Kopfzeilen aller Zwischenstaende in data/interim
# Zweck: Datei mit der Zuordnung Crate -> Eintrag finden (fehlt in 05_35_resolver_alt_neu.csv).
# Begruendung: Schnittstellen werden nachgelesen, nicht angenommen.
# Ausgabe: je CSV Zeilenzahl und Spaltennamen; keine Inhalte (P11).
for _p in sorted(K.interim.glob("*.csv")):
    _kopf = pd.read_csv(_p, nrows=0).columns.tolist()
    _n = sum(1 for _ in open(_p, encoding="utf-8", errors="replace")) - 1
    print(f"{_p.name}  ({_n} Zeilen)\n    {', '.join(_kopf)}")

In [ ]:
# Zelle 08 – Schluesselpruefung: Crate-Eintraege (NB02 Z.02) <-> Resolver-Ergebnis (NB05 Z.35)
# Zweck: Fuer status/zeige_vorschlaege je Crate muss jeder der 7.915 Crate-Eintraege genau
#        ein Resolver-Ergebnis finden.
# Begruendung: 390 Crate-Pfade fallen erst nach NFC zusammen (NB02); Merge nur auf eindeutigem
#        Schluessel (Befund NB05 Z.37: Vervielfachung).
# Ausgabe: Treffer je Schluesselvariante (exakt / NFC), Eindeutigkeit, Widersprueche; keine Pfade (P11).
import unicodedata

df_ce = pd.read_csv(K.interim / "02_02_crate_eintraege.csv", dtype=str)
df_res = pd.read_csv(K.interim / "05_35_resolver_alt_neu.csv", dtype=str, low_memory=False)
df_rc = df_res[df_res["quelle"] == "crates"]
print(f"Crate-Eintraege {len(df_ce)} | Resolver-Zeilen crates {len(df_rc)} | "
      f"davon eindeutige pfad {df_rc['pfad'].nunique()}")

nfc = lambda s: unicodedata.normalize("NFC", s) if isinstance(s, str) else s
for name, f in [("exakt", lambda s: s), ("NFC", nfc)]:
    r = df_rc.assign(_k=df_rc["pfad"].map(f))
    # Widerspruch = gleicher Schluessel, aber verschiedenes Ergebnis
    erg = r.groupby("_k")[["klasse_neu", "stufe_neu", "pfad_ziel_neu"]].nunique(dropna=False)
    widerspruch = int((erg > 1).any(axis=1).sum())
    schluessel = set(r["_k"])
    treffer = df_ce["pfad"].map(f).isin(schluessel)
    print(f"{name:6s}: Schluessel {len(schluessel)} | Widersprueche {widerspruch} | "
          f"Crate-Eintraege mit Treffer {int(treffer.sum())} von {len(df_ce)}")

In [ ]:
# Zelle 09 – Agent-Werkzeuge 5.1a auf echten Artefakten
# Zweck: status/finde_playlist/zeige_vorschlaege gegen bekannte Kennzahlen pruefen.
# Erwartung (NB05 Abschluss): Crates 167 Playlists, davon 130 vollstaendig;
#   uebrige Quellen zusammen 185 Playlists, davon 58 vollstaendig (leere Playlists fehlen hier,
#   weil sie keine Eintraege haben -> evtl. weniger).
# Ausgabe: Zaehlwerte; finde_playlist nur Anzahl/Score, keine Namen (P11).
from playlist_agent.agent import tools

db = tools.Datenbasis.aus_artefakten(K.interim)
print(f"Eintraege gesamt: {len(db.eintraege)}\n")
df_st = pd.DataFrame(tools.status(db)["je_quelle"])
print(df_st.to_string(index=False))
_ue = df_st[df_st["quelle"] != "crates"]
print(f"\nuebrige Quellen: {_ue['playlists'].sum()} Playlists, davon {_ue['vollstaendig'].sum()} vollstaendig")

_v = tools.zeige_vorschlaege(db, max_eintraege=5)
print(f"\nVorschlaege: {_v['pfade']} Pfade, {_v['eintraege']} Eintraege, je Stufe {_v['je_stufe']}")

# Selbsttest Suche: jeder echte Playlist-Name muss sich selbst als Treffer 1 finden
_namen = db.eintraege["playlist"].dropna().unique()
_ok = sum(tools.finde_playlist(db, n)["treffer"][0]["playlist"] == n for n in _namen)
print(f"finde_playlist Selbsttest: {_ok} von {len(_namen)} Namen finden sich selbst")

In [ ]:
# Zelle 10 – Diagnose finde_playlist: Namen, die sich nicht selbst als Treffer 1 finden
# Zweck: Einzelfall aus Z.09 (344 von 345) verstehen: Rest-Fehler der Rangfolge oder
#        echte Namensgleichheit nach Normalisierung (NFC, Gross/Klein)?
# Ausgabe: Name, normalisierte Form, Top-3-Treffer mit exakt/Score. Nur lokal angezeigt,
#        nicht gespeichert (echte Namen, P11; nbstripout entfernt Ausgaben vor dem Commit).
import unicodedata

for _n in _namen:
    _r = tools.finde_playlist(db, _n, max_treffer=3)
    if _r["treffer"][0]["playlist"] != _n:
        print(f"gesucht:    {_n!r}")
        print(f"normiert:   {unicodedata.normalize('NFC', _n).casefold().strip()!r}")
        print(f"eindeutig:  {_r['eindeutig']}")
        for _t in _r["treffer"]:
            _tn = unicodedata.normalize("NFC", _t["playlist"]).casefold().strip()
            print(f"  -> {_t['playlist']!r}  quelle={_t['quelle']}  exakt={_t['exakt']}  "
                  f"score={_t['score']}  normiert gleich={_tn == unicodedata.normalize('NFC', _n).casefold().strip()}")

In [ ]:
# Zelle 11 – Selbsttest finde_playlist mit Quelle
# Zweck: Mit Angabe der Quelle muss jede Playlist (Quelle + Name) sich selbst eindeutig finden.
# Erwartung: 352 von 352 (Summe Playlists aller Quellen, Z.09); Rueckfragen ohne Quelle: 3
#            (der Name aus Z.10 in Mac, Asus und Medion).
_pl = db.eintraege[["quelle", "playlist"]].drop_duplicates()
_ok = sum(
    (r := tools.finde_playlist(db, n, quelle=q))["eindeutig"] and r["treffer"][0]["playlist"] == n
    for q, n in _pl.itertuples(index=False))
_rueck = sum(not tools.finde_playlist(db, n)["eindeutig"] for n in _pl["playlist"])
print(f"mit Quelle eindeutig und richtig: {_ok} von {len(_pl)}")
print(f"ohne Quelle Rueckfrage noetig:    {_rueck} von {len(_pl)}")

In [ ]:
# Zelle 12 – Diagnose: Playlists, die ohne Quellenangabe eine Rueckfrage brauchen
# Zweck: Abweichung Z.11 (15 statt angekuendigt 3) aufklaeren; erwartete Ursache (Hypothese):
#        gleiche Namen in mehreren Quellen, v. a. foobar_asus/foobar_medion.
# Ausgabe: je normiertem Namen die Quellen und die Ursache (exakt gleich / in anderem Namen
#        enthalten). Nur lokal angezeigt, nicht gespeichert (echte Namen, P11).
_rf = _pl[[not tools.finde_playlist(db, n)["eindeutig"] for n in _pl["playlist"]]].copy()
_rf["normiert"] = _rf["playlist"].map(lambda s: unicodedata.normalize("NFC", s).casefold().strip())
_alle_norm = _pl["playlist"].map(lambda s: unicodedata.normalize("NFC", s).casefold().strip())
_rf["gleicher Name in Quellen"] = _rf["normiert"].map(
    lambda x: ", ".join(sorted(_pl.loc[_alle_norm == x, "quelle"])))
_rf["Ursache"] = ["gleicher Name" if (_alle_norm == x).sum() > 1 else "in anderem Namen enthalten"
                  for x in _rf["normiert"]]
print(_rf.groupby("Ursache").size().to_string(), "\n")
with pd.option_context("display.max_colwidth", 80, "display.width", 200):
    display(_rf.sort_values(["Ursache", "normiert", "quelle"])
               [["Ursache", "playlist", "quelle", "gleicher Name in Quellen"]])

### Befunde 5.1a – lesende Agent-Werkzeuge, 24.11.2026

**Modul:** `src/playlist_agent/agent/tools.py` – `status`, `finde_playlist(text, quelle)`,
`zeige_vorschlaege(playlist)`; JSON-Schema `WERKZEUGE`, Einstieg `rufe_auf` (unbekanntes Werkzeug /
falsche Argumente -> Fehlermeldung statt Ausnahme). Nur lesend (P2, P7); schreibende Aktionen folgen
ueber das Bestaetigungs-Gate (P9, Paket 5.2).

**Datenbasis:** Artefakte statt Stick. Crate-Eintraege (NB02 Z.02) + Resolver-Ergebnis (NB05 Z.35),
Schluessel exakter Pfad: 7.915/7.915 Treffer, 0 Widersprueche (Z.08). Merge mit `validate="m:1"`
und Abbruch bei fehlendem Ergebnis (Lehre NB05 Z.37).

**Pruefung gegen bekannte Kennzahlen (Z.09)**
| Groesse | erwartet (NB05) | gemessen |
|---|---|---|
| Crates: Playlists / vollstaendig | 167 / 130 | 167 / 130 |
| uebrige Quellen: Playlists / vollstaendig | 185 / 58 | 185 / 58 |
| Vorschlaege [Eintraege] | 1.214 | 1.214 |

**finde_playlist**
- Rangfolge: exakter Name > alle Woerter enthalten > Aehnlichkeit. Ohne Stufe 1 fanden sich nur
  313 von 345 Namen selbst (Name in laengerem Namen enthalten -> alphabetisch verloren); korrigiert.
- "eindeutig" je Playlist (Quelle + Name), nicht je Name.
- Mit Quellenangabe: 352/352 eindeutig und richtig (Z.11).
- Ohne Quellenangabe Rueckfrage bei 15/352 (4,3 %, Vollerhebung): 7 Namen in mehreren Quellen
  (Asus<->Medion, Medion<->Mac, Mac<->Musikordner), unabhaengig voneinander von Hand angelegt (Nutzer).
  Angekuendigt waren 3 – Zaehlung der Namensdubletten vorab versaeumt.

**Fuer Phase 6 (Sync):** die 7 Namensgruppen sind die Eingangsliste fuer den Inhaltsabgleich;
gleicher Name belegt keinen gleichen Inhalt.

In [ ]:
# Zelle 13 – Messung: Jahr aus Ordnername, Dateiname und tag_year
# Zweck: Abdeckung und Widersprueche der drei Jahresquellen messen, bevor eine Regel entsteht.
# Begruendung: datei_mtime und serato_hinzugefuegt sind laut Nutzer keine gute Quelle (24.11.2026).
# Suchmuster: 1950-2026 als eigenstaendige Zahl oder als Anfang eines 8-stelligen Datums (JJJJMMTT).
# Ausgabe: Zaehlungen; Einzelfaelle privat in reports/errors.
import re

df_bib = pd.read_csv(PROCESSED / "06_06_bibliothek.csv", dtype=str, low_memory=False)
_JAHR = re.compile(r"(?<!\d)(19[5-9]\d|20[0-2]\d)(?:[01]\d[0-3]\d)?(?!\d)")

def jahre(text) -> list[int]:
    if not isinstance(text, str):
        return []
    return sorted({int(j) for j in _JAHR.findall(text) if 1950 <= int(j) <= 2026})

df_j = pd.DataFrame({
    "pfad_relativ": df_bib["pfad_relativ"],
    "j_ordner": df_bib["datei_ordner"].map(lambda s: jahre(s.split("/")[-1] if isinstance(s, str) else None)),
    "j_datei": df_bib["datei_name"].map(jahre),
    "j_tag": df_bib["tag_year"].map(jahre),
    "tag_year_roh": df_bib["tag_year"],
})
for q in ["j_ordner", "j_datei", "j_tag"]:
    n = df_j[q].map(len)
    print(f"{q:9s}: mit Jahr {int((n > 0).sum()):5d} | davon mehrere Jahre {int((n > 1).sum()):4d}")

_alle = df_j[["j_ordner", "j_datei", "j_tag"]].apply(lambda r: set().union(*r), axis=1)
df_j["quellen"] = df_j[["j_ordner", "j_datei", "j_tag"]].apply(
    lambda r: "+".join(q[2:] for q, v in r.items() if v), axis=1)
df_j["widerspruch"] = _alle.map(len) > 1
print(f"\nohne jedes Jahr: {int((_alle.map(len) == 0).sum())} | Widerspruch (>1 verschiedenes Jahr): "
      f"{int(df_j['widerspruch'].sum())}")
print("\nKombination der Quellen [Dateien]:")
print(df_j["quellen"].replace("", "keine").value_counts().to_string())
print("\ntag_year Rohwerte ohne erkanntes Jahr (haeufigste 15):")
print(df_bib.loc[df_j["j_tag"].map(len).eq(0) & df_bib["tag_year"].notna(), "tag_year"]
      .value_counts().head(15).to_string())

_ = projekt.tabelle_speichern(df_j.loc[df_j["widerspruch"]].astype(str), "06_13_jahr_widersprueche",
                              ordner=K.err)

In [ ]:
# Zelle 14 – Jahresregel: Variante A (Ordner > Datei > Tag) gegen B (Ordner > Tag > Datei)
# Zweck: Regel fuer das Merkmal 'jahr' an den Daten entscheiden (Nutzerentscheidung).
# Beide: Rip-Datum (Form '(TT.MM.JJJJ hh-mm-ss)' im Ordnernamen) zaehlt nie als Jahr;
#        Ordner/Datei nur, wenn genau ein Jahr gefunden wird.
# Ausgabe: Zaehlungen je Herkunft; Faelle mit A != B privat in reports/errors zur Pruefung.
_RIP = re.compile(r"\d{1,2}\.\d{1,2}\.(?:19|20)\d{2}")

def jahre_ohne_rip(text) -> list[int]:
    return jahre(_RIP.sub(" ", text)) if isinstance(text, str) else []

_ord = df_bib["datei_ordner"].map(lambda s: jahre_ohne_rip(s.split("/")[-1]) if isinstance(s, str) else [])
_dat = df_bib["datei_name"].map(jahre_ohne_rip)
_tag = df_j["j_tag"]

def waehle(reihenfolge):
    out = []
    for quellen in zip(*[q for _, q in reihenfolge]):
        for (name, _), j in zip(reihenfolge, quellen):
            if len(j) == 1:
                out.append((j[0], name)); break
        else:
            out.append((None, "keine"))
    return out

_A = waehle([("ordner", _ord), ("datei", _dat), ("tag", _tag)])
_B = waehle([("ordner", _ord), ("tag", _tag), ("datei", _dat)])
df_jv = pd.DataFrame({"pfad_relativ": df_bib["pfad_relativ"],
                      "j_ordner": _ord, "j_datei": _dat, "j_tag": _tag,
                      "jahr_A": [a[0] for a in _A], "herkunft_A": [a[1] for a in _A],
                      "jahr_B": [b[0] for b in _B], "herkunft_B": [b[1] for b in _B]})
print("Rip-Datum entfernt bei Ordnern:", int((df_j["j_ordner"].map(len) != _ord.map(len)).sum()), "Dateien")
print("\nHerkunft A:", df_jv["herkunft_A"].value_counts().to_dict())
print("Herkunft B:", df_jv["herkunft_B"].value_counts().to_dict())
_diff = df_jv[df_jv["jahr_A"].astype(str) != df_jv["jahr_B"].astype(str)]
print(f"\nA != B: {len(_diff)} Dateien -> bitte pruefen: welches Jahr stimmt?")
_ = projekt.tabelle_speichern(_diff.astype(str), "06_14_jahr_a_gegen_b", ordner=K.err)

In [ ]:
# Zelle 15 – Gesamtliste aller Playlist-Eintraege (zweites Quelldokument neben der Bibliothek)
# Zweck: Eine Zeile je Eintrag aller 352 Playlists mit Aufloesungsergebnis, dauerhaft in
#        data/processed (Nutzerentscheidung 24.11.2026), statt nur im Speicher der Werkzeuge.
# Quelle: Crate-Eintraege (NB02 Z.02) + Resolver (NB05 Z.35), wie tools.Datenbasis (Z.08/Z.09).
# Erwartung: 18.204 Zeilen; Spalten quelle, playlist, position, pfad, klasse, stufe, pfad_ziel.
from playlist_agent.agent import tools

db = tools.Datenbasis.aus_artefakten(K.interim)
df_pe = db.eintraege
print(f"Zeilen {len(df_pe)} | Playlists {df_pe[['quelle', 'playlist']].drop_duplicates().shape[0]} | "
      f"Spalten {', '.join(df_pe.columns)}")
print(df_pe["klasse"].value_counts().to_string())
_ = projekt.tabelle_speichern(df_pe, "06_15_playlist_eintraege", ordner=PROCESSED)

In [ ]:
# Zelle 16 – plane_playlist_nach_kriterien auf der Masterliste
# Zweck: Werkzeug mit drei typischen Befehlen pruefen; nur Zaehlwerte (P11).
# Erwartung: Filterwirkung bewertung_min=5 = 547 (Z.06); Zeitraum 2025-2026 -> 0 Treffer mit Hinweis.
from playlist_agent.agent import tools

db = tools.Datenbasis.aus_artefakten(K.interim, PROCESSED)

faelle = {
    "a 5 Sterne Techno, max 2 h": dict(bewertung_min=5, stil=["techno"], gesamtdauer_max=7200,
                                       sortierung="zufall"),
    "b 5 Sterne letzte 2 Jahre":  dict(bewertung_min=5, jahr_von=2025, jahr_bis=2026),
    "c soft/happy 2005-2009":     dict(stimmung=["soft", "happy"], jahr_von=2005, jahr_bis=2009,
                                       sortierung="bewertung"),
}
for name, k in faelle.items():
    r = tools.rufe_auf(db, "plane_playlist_nach_kriterien", k)
    print(f"{name}: {r['stuecke']} Stuecke, {r['gesamtdauer_min']} min, plan {r['plan_id']}")
    print(f"   Filterwirkung {r['filterwirkung']}")
    print(f"   Hinweise {r['hinweise']}")
print(f"\nPlaene im Speicher: {len(db.plaene)}")

### Befunde 5.1b – planendes Werkzeug plane_playlist_nach_kriterien, 25.11.2026

**Grundlage:** zwei Quelldokumente in `data/processed` (privat):
`06_06_bibliothek.csv` (5.236 Musikstuecke, 139 Spalten) und `06_15_playlist_eintraege.csv`
(18.204 Eintraege, 352 Playlists).

**Merkmal Jahr (Entstehungsjahr: Erscheinungsjahr der CD bzw. Jahr des Sets)**
- Regel: Ordner (genau 1 Jahr, tiefster zuerst) > Datei-Datum > Tag > Datei (allein stehende Zahl) > Jahrzehnt.
- Beleg: Nutzerlabels bei Datei<>Tag-Widerspruch: Jahr in Datumsform im Dateinamen 14/14 richtig,
  allein stehende Zahl im Titel 0/3 (Tag richtig). Gesamt 30/31 gelabelte Faelle richtig
  (1 Fall mit Buchstabe o statt Ziffer 0, keine Sonderregel).
- Rip-Zeitpunkt '(TT.MM.JJJJ hh-mm-ss)' im Ordnernamen zaehlt nicht. datei_mtime und
  serato_hinzugefuegt sind keine Quelle (Nutzer).
- Ergebnis: Jahr 4.290 (82 %), nur Jahrzehnt 5, ohne 941; Spanne 1957-2022, Median 2006.

**Werkzeug-Regeln (Nutzerentscheidungen 25.11.2026)**
1. Mehrere Werte je Kriterium = ODER, verschiedene Kriterien = UND.
2. Gesamtdauer: der Reihe nach aufnehmen, bis das naechste Stueck nicht passt (kein Nachfuellen).
3. DJ-Treffer jeder Herkunft zaehlen; Treffer ohne Gold-Beleg werden im Hinweis gezaehlt.
4. Dubletten (gleicher Audio-Hash, sonst Datei-Hash) nur einmal, kuerzester Pfad.
- Ergebnis ist ein Plan (Vorschau): Plan-ID, Inhalts-Hash (SHA-256 der Pfadliste), Filterwirkung
  je Kriterium, Hinweise. Es wird nichts geschrieben (P9).

**Pruefung auf echten Daten (Z.16, Vorhersagen vorab notiert)**
| Fall | Stuecke | Dauer [min] | Befund |
|---|---|---|---|
| 5 Sterne Techno, max 2 h | 16 von 263 | 114,8 | Filterwirkung Bewertung 547 = Z.06 |
| 5 Sterne 2025-2026 | 0 | 0 | Hinweis "Jahre im Bestand 1957-2022" |
| soft/happy 2005-2009 | 142 | 1.140,9 | sehr gross -> Agent soll nach Grenze fragen (5.5) |

**Offen fuer 5.2:** Plaene liegen nur im Speicher und gehen beim Neustart verloren.

In [ ]:
# Zelle 17 – Gate end-to-end: Plan -> Vorschau -> Bestaetigung -> Ausfuehrung -> Protokoll
# Zweck: Ablauf P9 einmal vollstaendig pruefen. Ziel ist ein TESTORDNER in data/ (P7: echter Stick
#        erst nach Abnahme), Name nach Muster Playlist_*_foobar2000.
# Erwartung: 1. Lauf 'neu', 2. Lauf 'unveraendert'; falsche Bestaetigung -> abgelehnt; Ausgabe ohne Titel (P11).
from pathlib import Path

from playlist_agent.agent import gate, tools

PLAENE = K.wurzel / "data" / "plaene"
TESTZIEL = K.wurzel / "data" / "test_ziel" / "Playlist_Test_foobar2000"

db = tools.Datenbasis.aus_artefakten(K.interim, PROCESSED, plan_ordner=PLAENE)

# 1 Plan + Vorschau (das sieht spaeter der Nutzer im UI)
r = tools.rufe_auf(db, "plane_playlist_nach_kriterien",
                   dict(bewertung_min=5, stil=["techno"], gesamtdauer_max=7200, sortierung="zufall"))
print(f"Plan {r['plan_id']}: {r['stuecke']} Stuecke, {r['gesamtdauer_min']} min, Hash {r['inhalt_hash'][:12]}...")

# 2 Falsche Bestaetigung muss abgelehnt werden
try:
    gate.fuehre_aus(PLAENE, r["plan_id"], "0" * 64, "Test 5 Sterne Techno", TESTZIEL, "D:", "asus")
    print("FEHLER: falsche Bestaetigung wurde akzeptiert")
except gate.AusfuehrungAbgelehnt as e:
    print(f"abgelehnt wie erwartet: {e}")

# 3 Bestaetigung = genau der gezeigte Hash -> Ausfuehrung, zweimal
for lauf in (1, 2):
    a = gate.fuehre_aus(PLAENE, r["plan_id"], r["inhalt_hash"], "Test 5 Sterne Techno", TESTZIEL, "D:", "asus")
    print(f"Lauf {lauf}: status {a['status']}, {a['eintraege']} Eintraege, Integritaet {a['integritaet']}")
    print(f"   Protokoll: {'/'.join(Path(a['protokoll']).parts[-4:])}")
print(f"\nPlaene gespeichert: {len(gate.liste_plaene(PLAENE))}")

### Befunde 5.2 – Bestaetigungs-Gate, 25.11.2026

**Ablauf (P9):** Plan -> Vorschau -> Bestaetigung -> Ausfuehrung -> Protokoll.
`gate.fuehre_aus` ist kein LLM-Werkzeug (Test: keine Schreib-/Ausfuehrungsfunktion in
`tools.WERKZEUGE`); Aufruf nur durch Notebook/UI nach ausdruecklicher Nutzerbestaetigung.

**Plaene dauerhaft** (Nutzerentscheidung 25.11.2026): JSON in `data/plaene/` (privat), nie veraendert;
gleiche Plan-ID mit anderem Inhalt -> Abbruch; beim Laden Inhalts-Hash neu berechnet ->
manipulierte Datei wird nie ausgefuehrt; Plan-ID gegen festes Muster (kein Pfad von aussen).

**Pruefungen vor dem Schreiben** (eine verletzt -> nichts geschrieben)
1. Plan intakt (Hash) 2. bestaetigter Hash = Plan-Hash 3. Zielordner `Playlist_*_foobar2000` (P3)
4. Integritaet kritisch -> nur mit ausdruecklichem `trotz_warnung` (P12)
5. gleichnamige Playlist mit anderem Inhalt -> nur mit `ueberschreiben`, dann Backup (P3); leerer Plan -> abgelehnt.
Jede Ausfuehrung: neues Protokoll in `data/plaene/ausgefuehrt/`.

**Pruefung (Z.17, Testordner in data/, nicht der Stick, P7)**
| Schritt | Ergebnis |
|---|---|
| Plan (5 Sterne Techno, max 2 h) | 16 Stuecke, 114,8 min, Plan-ID wie Z.16 (reproduzierbar) |
| falsche Bestaetigung | abgelehnt, nichts geschrieben |
| Ausfuehrung 1 / 2 / 3 | neu / unveraendert / unveraendert (idempotent) |

**Offen:** Integritaetspruefung an das Gate anbinden, sobald das UI den Stick verankert (Phase 7);
Pflichtbericht (P4) fuer Agent-Aktionen in Phase 7 in `report` integrieren.

In [ ]:
# Zelle 19 – Luecken a-f, i: Testset-Befehle mit den neuen Kriterien auf der Masterliste
# Zweck: Erwartete Argumente aus dem Gold-Label (Z.18) direkt ausfuehren; nur Zaehlwerte (P11).
# Argumente stammen aus data/gold/06_18_befehle_erwartung.csv (keine Namen im Code).
import json
from pathlib import Path

from playlist_agent.agent import tools

db = tools.Datenbasis.aus_artefakten(K.interim, PROCESSED)
df_gold = pd.read_csv(K.gold / "06_18_befehle_erwartung.csv", dtype=str, keep_default_na=False,
                      encoding="utf-8-sig")
zusatz = {"5": {"max_je_person": 2}}          # Nr. 5 'verschiedene Interpreten' -> Vielfalt

for nr in ["3", "5", "8", "9", "23"]:
    z = df_gold[df_gold["nr"] == nr].iloc[0]
    werkzeug = z["werkzeug"].split("->")[-1].strip()
    args = {**json.loads(z["argumente"] or "{}"), **zusatz.get(nr, {})}
    r = tools.rufe_auf(db, werkzeug, args)
    if "fehler" in r:
        print(f"Nr. {nr}: FEHLER {r['fehler']}")
    elif werkzeug == "finde_titel":
        print(f"Nr. {nr} {werkzeug}: {r['stuecke']} Stuecke, in Playlists: "
              f"{[len(x['playlists']) for x in r['liste']]}")
    else:
        print(f"Nr. {nr} {werkzeug}: {r['stuecke']} Stuecke, {r['gesamtdauer_min']} min")
        print(f"   Filterwirkung {r['filterwirkung']}")
        print(f"   Hinweise {r['hinweise']}")

### Befunde 5.1c (Teil 1) – Testset und Luecken, 25.11.2026

**Testset (Gold-Label):** 38 Befehle im Wortlaut des Nutzers (`data/gold/06_18_befehle_nutzer.txt`),
erwartete Reaktion je Befehl vom Nutzer geprueft: 38/38 bestaetigt (`06_18_befehle_erwartung.csv`, privat).
Verteilung: Werkzeug 18 | Rueckfrage 9 | Luecke 7 | Ablehnung 4.

**Neue Regel aus der Nutzerpruefung – Personenrolle:** Person ohne Rolle im Befehl -> Rueckfrage
"als DJ (Sets) oder als Interpret/Komponist (eigene Stuecke)?"; ohne Rueckfrage nur bei ausdruecklicher Rolle.

**Luecken (Werkzeug-Abdeckung, gemessen vor jedem LLM)**
| Luecke | Entscheidung | Umsetzung |
|---|---|---|
| a Ausschluss | geschlossen | stimmung_nicht, stil_nicht |
| b Komponist | geschlossen | komponist (tag_composer, serato_komponist) |
| c BE markiert | geschlossen | be_markiert |
| d Ordner | geschlossen | ordner (Teilwort, * erlaubt) |
| e Titelsuche | geschlossen | Werkzeug finde_titel (mit Playlists je Stueck) |
| f Vielfalt | geschlossen | max_je_person (Interpret, sonst DJ) |
| i Mindest-Gesamtdauer | geschlossen | gesamtdauer_min (nur Hinweis, nicht erzwungen) |
| g Begriffe ausserhalb des Vokabulars | bleibt offen | Kern der Messung Regeln vs. LLM (5.7) |
| h Verlauf "seit letzter Woche" | spaeter | Manifeste je Lauf aufbewahren (Phase 7) |

**Pruefung auf echten Daten (Z.19, Argumente aus dem Gold-Label, keine Namen im Code)**
| Nr. | Stuecke | Dauer [min] | Befund |
|---|---|---|---|
| 3 | 18 von 186 | 228,9 (Grenze 240) | 14 Dubletten ausgelassen |
| 5 | 28 | 185,6 | Vielfalt: 20 ausgelassen (max 2 je Person) |
| 8 | 40 | 256,8 | Filterwirkung be_markiert 585 / 5 Sterne 547 = Z.06 |
| 9 | 11 | 75,6 | Ordner-Muster trifft 25 Dateien |
| 23 | 2 | – | beide Stuecke in je 2 Playlists |

In [ ]:
# Zelle 20 – Werkzeuge 5.1c Teil 2 mit Testset-Argumenten (Gold-Label Z.18); nur Zaehlwerte (P11)
import json

from playlist_agent.agent import tools

db = tools.Datenbasis.aus_artefakten(K.interim, PROCESSED)
df_gold = pd.read_csv(K.gold / "06_18_befehle_erwartung.csv", dtype=str, keep_default_na=False,
                      encoding="utf-8-sig")
arg = lambda nr: json.loads(df_gold.loc[df_gold["nr"] == nr, "argumente"].iat[0] or "{}")

print("status:", tools.status(db)["stick"])

r = tools.zeige_beschaedigte_ordner(db)
print(f"\nbeschaedigte Ordner: {r['ordner']} | Eintraege mit Praefix-Treffer "
      f"{sum(x['eintraege'] for x in r['liste'])} | album_beschaedigt gesamt {r['album_beschaedigt_eintraege']}")

for nr in ["11", "12", "28", "29"]:
    r = tools.finde_ordner(db, **arg(nr))
    print(f"Nr. {nr} finde_ordner: eindeutig {r['eindeutig']}, Treffer {len(r['treffer'])}, "
          f"bester exakt {r['treffer'][0]['exakt']}, score {r['treffer'][0]['score']}")

r = tools.plane_create_from_folder(db, **arg("10"))
print(f"\nNr. 10 create_from_folder: {r.get('stuecke')} Stuecke, Regel {r.get('reihenfolge')}, "
      f"{r.get('gesamtdauer_min')} min, Fehler {r.get('fehler')}")

for nr in ["19", "20", "13"]:
    a = arg(nr)
    f = tools.finde_playlist(db, a.get("playlist") or a.get("text"), quelle=a.get("quelle"))
    if not f["eindeutig"]:
        print(f"Nr. {nr}: nicht eindeutig, {len(f['treffer'])} Treffer (Rueckfrage)"); continue
    name, quelle = f["treffer"][0]["playlist"], f["treffer"][0]["quelle"]
    if nr == "13":
        p = tools.plane_update(db, name, quelle)
        print(f"Nr. 13 plane_update: {p['stuecke']} geschrieben, ausgelassen {p['ausgelassen']}")
    else:
        s = tools.zeige_playlist_status(db, name, quelle)
        print(f"Nr. {nr} status: {s['eintraege']} Eintraege {s['je_klasse']}, vollstaendig {s['vollstaendig']}")

### Befunde 5.1c (Teil 2) – restliche Werkzeuge, 27.11.2026

**Neu:** `finde_ordner`, `plane_create_from_folder` (Reihenfolge aus `ordner.py`), `zeige_playlist_status`,
`zeige_beschaedigte_ordner`, `plane_update` (nur sichere Eintraege + Streams, P5), `status` mit
Stick-Verankerung (Anker vom Aufrufer gesetzt; Werkzeuge lesen den Stick nicht selbst).
Plan-Erzeugung zentral (`_neuer_plan`): alle planenden Werkzeuge speichern ueber das Gate.

**Pruefung (Z.20, Argumente aus dem Gold-Label)**
| Befehl | Ergebnis | Erwartung erfuellt |
|---|---|---|
| status | "Stick nicht verankert" | ja (Anker erst Phase 7) |
| beschaedigte Ordner | 23 Ordner; 1.164 von 1.202 Eintraegen per Praefix | ja (Rest = Praefix-Ergaenzung/lose, NB05 Z.23) |
| Nr. 11, 12 Ordner suchen | eindeutig, exakter Treffer | ja |
| Nr. 28, 29 Ordner suchen | nicht eindeutig -> Rueckfrage | ja (Gold: Rueckfrage) |
| Nr. 10 Ordner-Playlist | 93 Stuecke, Regel dateiname, 701,6 min | ja |
| Nr. 19 / 20 Status | vollstaendig / 107 von 283 album_beschaedigt | ja |
| Nr. 13 Aktualisieren | 12 geschrieben, 3 ausgeschlossen (kein Audio) | ja |

**Offen:** `check_integrity` (Stick neu einlesen und mit Manifest vergleichen) -> mit integrity.py;
Sync-Werkzeuge (`plane_sync`, `vergleiche_playlists`, `vergleiche_laptops`) -> Phase 6.

In [ ]:
# Zelle 21 – check_integrity auf dem echten Stick (lesend, P1/P7; Ausnahme Hash/Tags, Entscheidung 07.11.)
# Zweck: erster Lauf gegen das Startmanifest (NB04 Z.08); legt data/manifeste/ an.
# Erwartung: Datenbasis seit 15.11. unveraendert (Nutzer) -> 0 kritische Befunde.
from dataclasses import asdict

from playlist_agent import anchor
from playlist_agent.agent import tools

db = tools.Datenbasis.aus_artefakten(K.interim, PROCESSED, manifest_ordner=K.wurzel / "data" / "manifeste")
a = anchor.verankere(K.cfg)
db.anker = {**asdict(a), "wurzel": str(a.wurzel)}
print("status:", tools.status(db)["stick"])

r = tools.rufe_auf(db, "check_integrity", {})
for k in ["verglichen_mit", "dateien", "neu_gehasht", "kritisch", "neues_manifest", "hinweis"]:
    print(f"{k}: {r.get(k, r.get('fehler'))}")
print("zusammenfassung:", r.get("zusammenfassung"))

### Befunde 5.1c (Teil 3) – Integritaetspruefung als Werkzeug, 27.11.2026

**check_integrity (P12):** Stick einlesen (`index.scanne`), Manifest erstellen (Hash nur bei geaenderter
Groesse/mtime), Vergleich mit Basis (`integrity.vergleiche_manifeste`). Basis = neuestes
`data/manifeste/manifest_<Zeit>.csv`, beim ersten Lauf Startmanifest NB04 Z.08.
- ohne kritischen Befund: neues Manifest wird Basis
- mit kritischem Befund: nur `kandidat_<Zeit>.csv`; Basis erst nach Bestaetigung (`uebernehme_kandidat`,
  kein LLM-Werkzeug) -> ein Schaden wird nicht unbemerkt zur neuen Wahrheit (Entscheidung a)
- Test mit Schadensmuster 06.02.2020 (Ordner -> 0-Byte-Datei): kritisch erkannt, nur Kandidat.
- `rufe_auf` sperrt Argumente mit `_` (interne Test-Haken nicht von aussen erreichbar).

**Erster echter Lauf (Z.21):** Stick D: am Asus, Laufwerk wie konfiguriert; 7.386 Dateien,
0 neu gehasht, alle Befundtypen 0 -> 0 Fehlalarme bei unveraendertem Stick (Ziel Experimentdesign).

**Stand Werkzeuge:** 12 LLM-Werkzeuge (lesend: status, finde_playlist, finde_ordner, finde_titel,
zeige_vorschlaege, zeige_playlist_status, zeige_beschaedigte_ordner, check_integrity; planend:
plane_playlist_nach_kriterien, plane_create_from_folder, plane_update); Ausfuehrung nur ueber das Gate.
Offen: Sync-Werkzeuge (Phase 6).

## Abschluss Notebook 06 – Bibliothek, Agent-Werkzeuge, Bestaetigungs-Gate (Phase 5.0–5.2), 27.11.2026

**Ergebnis**
- Zwei Quelldokumente (privat, `data/processed`): `06_06_bibliothek.csv` (5.236 Musikstuecke, 114 Roh- +
  25 Merkmalsspalten mit Herkunft) und `06_15_playlist_eintraege.csv` (18.204 Eintraege, 352 Playlists).
- Merkmale: BE/BA, Stimmung/Stil, Bewertung (Konflikt -> Nutzer), DJ getrennt von Interpret, Jahr/Jahrzehnt;
  Regeln vom Nutzer entschieden, Genre-Zerlegung und Jahresregel gegen Nutzerlabels geprueft (Jahr 30/31).
- 12 LLM-Werkzeuge (lesend/planend), JSON-Schema, einziger Einstieg `rufe_auf`.
- Gate: Plaene dauerhaft und manipulationssicher; Ausfuehrung kein LLM-Werkzeug, 5 Pruefungen vor dem Schreiben.
- Integritaet als Werkzeug: erster echter Lauf 0 Befunde, 0 Fehlalarme.
- Testset (Gold-Label): 38 Befehle im Nutzerwortlaut, Erwartung 38/38 vom Nutzer bestaetigt;
  Regel Personenrolle neu; Luecken a–f, i geschlossen, g bleibt Messgegenstand, h spaeter.
- Tests: 186 -> 312 (pytest, CI Windows).

**Lehren**
- Roherfassung gehoert vollstaendig in die EDA (hier nachgeholt als eigener Erfassungsschritt).
- Zu jeder Zaehlung die Einzelfaelle zeigen; jede CSV meldet Pfad und Zeilenzahl.
- Angekuendigte Zahlen vorab nachrechnen (Fehlprognose Rueckfragen 3 statt 15: Namensdubletten nicht gezaehlt).
- Nach Ersetzen einer Datei in `src/` Kernel neu starten; jede Zelle importiert, was sie braucht.

**Weiter in `06b_agent_messung.ipynb`:** 5.3 Testset formalisieren (+ synthetische Varianten),
5.4 Baselines A1/A2, 5.5 Agent-Loop, 5.6 Modell-Screening, 5.7 Messung + MLflow, 5.8 Sicherheit, 5.9 Abschluss.